# RQ7.1 — Model Compression: Seed 14

Compare the effects of quantisation, structured pruning and knowledge distillation on MobileNetV3-Small and EfficientNet-B0.

| Setting | Value |
|---|---|
| Training seed | 42 |
| Batch size | 64 |
| Classes | 38 |
| Train / validation / test | 38,147 / 5,448 / 10,704 |
| Baseline training | 3 head-training + 17 fine-tuning epochs |
| Pruning recovery | 5 epochs per candidate |
| CE and KD students | 20 epochs each |
| Source snapshot | Same hybrid-v2 bundle as Kaggle seed 14, version 364832930 |

Run sections 1–5 with a GPU runtime. After preflight passes, enable training as described in section 6. Each execution uses a new timestamped output directory. The matching Kaggle seed-14 execution completed in version 364832930. Results and provenance are indexed in results/RQ7_1_Model_Compression_Seed14/.


## 1. Configuration

Confirm the seed and Drive paths. Keep `RUN_FULL=False` until preflight has passed. Do not rerun this configuration cell during an active experiment.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, shutil, subprocess, zipfile, threading
IS_KAGGLE=Path('/kaggle/working').is_dir()
RUN_FULL=False
SEEDS=[14]
assert len(SEEDS) == 1 and len(set(SEEDS))==len(SEEDS)
assert set(SEEDS) <= {14,27,42}
if IS_KAGGLE:
    BUNDLE=Path('/kaggle/input/datasets/makungundlovu/cos801-rq71-hybrid-v2-source/rq71_hybrid_v2_source.zip.bin')
    IMAGE_SOURCE=Path('/kaggle/input/datasets/makungundlovu/cos801-rq7-1-audited-plantvillage-archive/rq71_plantvillage')
    BASE=Path('/kaggle/working')
    BACKUP_BASE=None
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT=Path('/content/drive/MyDrive/COS801_Project/rq7.1-model-compression')
    BUNDLE=DRIVE_PROJECT/'rq71_hybrid_v2_source.zip'
    IMAGE_SOURCE=Path('/content/drive/MyDrive/COS801_data/rq71_plantvillage.tar')
    BASE=Path('/content')
    BACKUP_BASE=DRIVE_PROJECT/'results'/'colab_hybrid_seed14'
assert BUNDLE.is_file(), f'Upload/attach the NEW bundle and correct BUNDLE: {BUNDLE}'
assert IMAGE_SOURCE.exists(), f'Correct IMAGE_SOURCE: {IMAGE_SOURCE}'
ROOT=BASE/datetime.now(timezone.utc).strftime('rq71_hybrid_v2_%Y%m%dT%H%M%S_%fZ')
ROOT.mkdir(exist_ok=False)
BACKUP=BACKUP_BASE/ROOT.name if BACKUP_BASE else None
if BACKUP: BACKUP.mkdir(parents=True,exist_ok=False)
os.environ['PYTHONUNBUFFERED']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
print('Platform:', 'Kaggle' if IS_KAGGLE else 'Colab', 'Seeds:',SEEDS,flush=True)
print('Local output:',ROOT,'Persistent backup:',BACKUP,flush=True)


## 2. Source verification

Verify the source archive and its file inventory against the pinned SHA256 fingerprints. The source snapshot is identical to the Kaggle hybrid experiment.


In [ ]:
EXPECTED='8bf0c2a502caf94302ff310bf7b25756d5239aff1e95d02924591503550fa3b2'
local_bundle=ROOT/BUNDLE.name
shutil.copyfile(BUNDLE,local_bundle)
assert hashlib.sha256(local_bundle.read_bytes()).hexdigest()==EXPECTED, 'Wrong source bundle'
with zipfile.ZipFile(local_bundle) as archive:
    for member in archive.infolist():
        assert (ROOT/member.filename).resolve().is_relative_to(ROOT.resolve()), 'Unsafe archive path'
    assert archive.testzip() is None, 'Corrupt source ZIP'
    archive.extractall(ROOT)
PROJECT=ROOT/'rq71_full_source'
inventory=json.loads((PROJECT/'source_inventory.json').read_text())
for rel,expected in inventory['files_sha256'].items():
    path=(PROJECT/rel).resolve()
    assert path.is_relative_to(PROJECT.resolve())
    assert hashlib.sha256(path.read_bytes()).hexdigest()==expected, f'Source mismatch: {rel}'
(ROOT/'source_provenance.json').write_text(json.dumps(dict(bundle_sha256=EXPECTED,seeds=SEEDS,split_version='hybrid-v2',inventory=inventory),indent=2))
def run(args):
    print('$',' '.join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),cwd=PROJECT,check=True)
print('Verified source files:',len(inventory['files_sha256']),flush=True)


## 3. Environment

Verify CUDA availability, install the scoped dependencies and record hardware and package versions. Retain the runtime's compatible PyTorch/torchvision pair. Run the protocol integration checks before data staging.


In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU runtime required; stop and select one'
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,flush=True)
run([sys.executable,'-m','pip','install','-r',PROJECT/'requirements.txt'])
run([sys.executable,'-u','check_environment.py','--require-cuda','--output',ROOT/'environment.json'])
with (ROOT/'pip_freeze.txt').open('w') as handle:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=handle,check=True)
run([sys.executable,'-u','test_hybrid_integration.py'])


## 4. Dataset staging

Copy the image archive to runtime-local storage. Stage and hash-check all 54,299 manifest images according to the revised assignment table. Original Drive data remain unchanged; existing staging destinations are not overwritten.


In [ ]:
source=IMAGE_SOURCE
if not IS_KAGGLE and IMAGE_SOURCE.is_file():
    source=ROOT/IMAGE_SOURCE.name
    print('Copying image archive once from Drive...',flush=True)
    shutil.copyfile(IMAGE_SOURCE,source)
DATA=ROOT/'images'
MANIFESTS=PROJECT/'splits_hybrid_v2'
run([sys.executable,'-u','stage_hybrid_v2.py','--source',source,'--output',DATA,'--manifest-dir',MANIFESTS])


## 5. Preflight validation

Validate the reviewed manifests, split counts, image hashes and experiment configuration. This section does not start training.


In [ ]:
COMMAND=[sys.executable,'-u','run_three_seeds.py','--config',str(PROJECT/'configs/full.json'),'--data-dir',str(DATA),'--manifest-dir',str(MANIFESTS),'--output-root',str(ROOT/'results'),'--split-version','hybrid-v2','--seeds',*map(str,SEEDS)]
run(COMMAND)
PREFLIGHT_PASSED=True


## 6. Training and backup

After section 5 passes, run `RUN_FULL=True` in a new code cell, then run the cell below. Do not rerun sections 1–4 in the same session.

The pipeline runs linear probes, both baselines, compression and selected-model analyses. Accuracy-gate rejections are retained as negative results. Unexpected errors stop execution.

Colab results are written under `/content/rq71_hybrid_v2_<timestamp>/results` and copied to `MyDrive/COS801_Project/rq7.1-model-compression/results/colab_hybrid_seed14/<timestamp>` every five minutes during training and on normal exit. Backup does not implement checkpoint resume and cannot guarantee recovery after abrupt runtime loss. Verify the final backup before ending the session.


In [ ]:
if not RUN_FULL:
    print('Training disabled. Preflight completed; no models trained.',flush=True)
else:
    assert PREFLIGHT_PASSED
    def backup_snapshot():
        if BACKUP is None: return
        for parent in (ROOT/'results', ROOT):
            files=parent.rglob('*') if parent.name=='results' else parent.glob('*')
            for file in files:
                if not file.is_file() or (parent==ROOT and file.suffix not in ('.json','.txt','.log')): continue
                before=file.stat()
                destination=BACKUP/file.relative_to(ROOT)
                if destination.exists() and destination.stat().st_size==before.st_size and destination.stat().st_mtime_ns>=before.st_mtime_ns: continue
                destination.parent.mkdir(parents=True,exist_ok=True)
                temporary=destination.with_name(destination.name+'.copying')
                shutil.copy2(file,temporary)
                after=file.stat()
                if (before.st_size,before.st_mtime_ns)==(after.st_size,after.st_mtime_ns): temporary.replace(destination)
    stop=threading.Event()
    def periodic_backup():
        while not stop.wait(300):
            try:
                backup_snapshot()
                print('[backup] snapshot completed',flush=True)
            except Exception as exc: print('[backup] warning:',exc,flush=True)
    worker=threading.Thread(target=periodic_backup,daemon=True) if BACKUP else None
    if worker: worker.start()
    try:
        with (ROOT/'launcher.log').open('w') as log:
            with subprocess.Popen(COMMAND+['--execute'],cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
                for line in process.stdout:
                    print(line,end='',flush=True); log.write(line); log.flush()
                code=process.wait()
        if code: raise RuntimeError(f'Launcher exit {code}; preserve logs and inspect before retrying')
    finally:
        stop.set()
        if worker: worker.join(timeout=30)
        if not worker or not worker.is_alive(): backup_snapshot()
        else: print('Backup still active; verify before closing',flush=True)


## 7. Results and figures

Inspect each run manifest and final comparison table before interpreting results. The complete figure index includes training histories and any generated calibration, risk–coverage and compression plots. Choose figure indices in `SHOW_FIGURES` to display them.


In [ ]:
import pandas as pd
from IPython.display import display, Image
for path in sorted((ROOT/'results').glob('run_*/seed_*/full_run_manifest.json')):
    print(path.relative_to(ROOT)); display(json.loads(path.read_text()))
for path in sorted((ROOT/'results').glob('run_*/seed_*/final_comparison.csv')):
    print(path.relative_to(ROOT)); display(pd.read_csv(path))
FIGURES=sorted((ROOT/'results').rglob('*.png'))
for index,path in enumerate(FIGURES): print(index,path.relative_to(ROOT))
print('Figure count:',len(FIGURES),'Local results:',ROOT/'results','Drive backup:',BACKUP)


In [ ]:
SHOW_FIGURES=[]
for index in SHOW_FIGURES:
    print(FIGURES[index].relative_to(ROOT))
    display(Image(filename=str(FIGURES[index])))


## 8. Reproducibility and limitations

- Aggregate seed 14 with seeds 27 and 42 only after checking source, split and configuration fingerprints and software environments.
- Compare deployment latency on a common CPU target; hosted CPU measurements are proxies, not low-cost-device validation.
- Known author leaf IDs and exact image duplicates are disjoint across partitions. Unresolved leaf identities and unscreened near duplicates remain limitations.
- Test membership is fixed. Historical test results were inspected before the validation-protocol revision; disclose this and do not pool legacy and hybrid results as equivalent repetitions.
- Review both models' stage statuses, accuracy-gate outcomes, tables, histories, analysis artifacts and backups before declaring the experiment complete.
